# Import library

In [80]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import savgol_filter # Savitzky-Golay (SG) Derivatives
from sklearn.decomposition import PCA

# Import and explore data

In [81]:
df = pd.read_csv('coffee_barley_nir.csv',sep=";")

In [ ]:
df["set"].unique

# Data has first column for training and testing set so lets seperate data into two set

In [85]:
df_train = df[df["set"] == "training"]

In [86]:
df_test = df[df["set"] == "external_test"]

# number of total rows did not match total row = 158, train is 117 and test is 11 as df.unique showed all value but not displayed 

In [91]:
df_inttest = df[df["set"]=="internal_test"]

# Write data to three seperate csv file

In [94]:
df_test.to_csv("test_data.csv", sep=";", index=False)

In [95]:
df_train.to_csv("train_data.csv", sep=";", index=False)

In [96]:
df_inttest.to_csv("int_test_data.csv", sep=";", index=False)

# Lets plot test data 

# Frist need to remove index and frist two column i.e. set and % barley

In [97]:
wavelength = df.columns[3:].astype(float)

In [ ]:
wavelength

In [99]:
x_train = df_train.iloc[2:,3:].to_numpy()
y_train = df_train.iloc[2:,2].to_numpy()

## Lest plot data


In [ ]:
#ax.autoscale()
Fig, ax = plt.subplots(figsize=(11, 5.5), dpi=100)
ax.plot(wavelength, x_train.T)
ax.set_xlabel('wavelength (cm-1)', fontsize=12)
ax.set_ylabel('intensity', fontsize=12)
ax.set_title('Nir data train set', fontsize=14, fontweight='bold')
ax.grid(True, alpha=0.3)

## Lets apply some prepocessing
### Data preprocessing 
#### 1)Standard Normal Variate (SNV)SNV operates row-by-row (sample-by-sample). For each individual spectrum , it subtracts its mean and divides by its standard deviation :
#### Sitzky-Golay (SG) Derivatives Savitzky-Golay uses local polynomial regression across a moving window of wavelengths. It simultaneously smooths noise and calculates numerical derivatives:
1st Derivative (deriv=1): Eliminates flat baseline offsets and highlights slope changes

2nd Derivative (deriv=2): Eliminates linear baseline slope/tilt and resolves overlapping absorption bands.

Key Rule for savgol_filter: window_length must be an odd number (e.g., 9, 15, 21) and larger than polyorder (typically 2 or 3). A wider window increases smoothing but reduces sharp peak heights.


### Defining function for SNV

In [103]:
def apply_snv(X_data):
    """
    Applies Standard Normal Variate (SNV) transformation.
    X_data: NumPy array of shape (n_samples(row), n_wavelengths(column))
    """
    mean = np.mean(X_data, axis=1, keepdims=True)
    std = np.std(X_data, axis=1, keepdims=True)
    return (X_data - mean) / std

In [104]:
x_train_snv = apply_snv(x_train)

In [105]:
# 1st Derivative (window_length must be an ODD integer)
x_train_sg1 = savgol_filter(x_train, window_length=15, polyorder=2, deriv=1, axis=1)

# 2nd Derivative
x_train_sg2 = savgol_filter(x_train, window_length=15, polyorder=2, deriv=2, axis=1)

# Combined Pipeline: SNV followed by 1st Derivative (Very common in chemometrics!)
x_train_snv_sg1 = savgol_filter(x_train_snv, window_length=15, polyorder=2, deriv=1, axis=1)

# Combined Pipeline: SNV followed by 2st Derivative (Very common in chemometrics!)
x_train_snv_sg2 = savgol_filter(x_train_snv, window_length=15, polyorder=2, deriv=2, axis=1)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 9), dpi=100)

plots = [
    (x_train, '1. Raw Spectra', axes[0, 0]),
    (x_train_snv, '2. SNV Preprocessed', axes[0, 1]),
    (x_train_sg1, '3. Savitzky-Golay 1st Derivative', axes[1, 0]),
    (x_train_snv_sg1, '4. Combined (SNV + SG 1st Deriv)', axes[1, 1])
]

for data, title, ax in plots:
    # Quick line plot colored by barley percentage y
    for i in range(len(data)):
        ax.plot(wavelength, data[i], alpha=0.5, linewidth=0.8)
    ax.set_title(title, fontweight='bold')
    ax.set_xlabel('Wavelength')
    ax.set_ylabel('Intensity')
    ax.grid(True, linestyle=':', alpha=0.6)

plt.tight_layout()
plt.show()

In [ ]:
x_prep = x_train_snv_sg1

# Fit PCA Model

n_components = 5
pca = PCA(n_components=n_components)

# Scores: Coordinates of samples in PC space (n_samples, n_components)
scores = pca.fit_transform(x_prep)

# Loadings: Contribution of each wavelength to each PC (n_components, n_wavelengths)
loadings = pca.components_

# Variance explained by each Principal Component
exp_var = pca.explained_variance_ratio_ * 100

print("--- Explained Variance Ratio per PC ---")
for i, var in enumerate(exp_var):
    print(f"  PC{i+1}: {var:.2f}%  (Cumulative: {np.sum(exp_var[:i+1]):.2f}%)")


# PCA Scores Plot (PC1 vs PC2)

fig, ax = plt.subplots(figsize=(8.5, 6), dpi=100)

# Scatter plot colored continuously by barley concentration percentage (y)
scatter = ax.scatter(
    scores[:, 0], scores[:, 1], 
    c=y_train, cmap='viridis', edgecolors='k', linewidths=0.5, s=65, alpha=0.9
)

cbar = fig.colorbar(scatter, ax=ax)
cbar.set_label('Barley Concentration (%)', fontsize=11, fontweight='bold')

ax.set_xlabel(f'PC1 ({exp_var[0]:.2f}% Variance)', fontsize=11, fontweight='bold')
ax.set_ylabel(f'PC2 ({exp_var[1]:.2f}% Variance)', fontsize=11, fontweight='bold')
ax.set_title('PCA Scores Plot (PC1 vs PC2)', fontsize=13, fontweight='bold')

# Reference crosshairs at origin (0,0)
ax.axhline(0, color='gray', linestyle='--', linewidth=0.8)
ax.axvline(0, color='gray', linestyle='--', linewidth=0.8)
ax.grid(True, linestyle=':', alpha=0.6)

plt.tight_layout()
plt.show()


# PCA Loadings Plot (PC1 & PC2 across Wavelengths)

fig, ax = plt.subplots(figsize=(11, 5), dpi=100)

ax.plot(wavelength, loadings[0], label=f'PC1 Loading ({exp_var[0]:.2f}%)', color='#1f77b4', linewidth=1.5)
ax.plot(wavelength, loadings[1], label=f'PC2 Loading ({exp_var[1]:.2f}%)', color='#d62728', linewidth=1.5)

ax.axhline(0, color='black', linestyle='--', linewidth=0.8)
ax.set_xlabel('Wavelength / Wavenumber', fontsize=11, fontweight='bold')
ax.set_ylabel('Loading Value', fontsize=11, fontweight='bold')
ax.set_title('PCA Loadings Plot (Spectral Feature Importance)', fontsize=13, fontweight='bold')
ax.legend(fontsize=10, loc='best')
ax.grid(True, linestyle=':', alpha=0.6)

plt.tight_layout()
plt.show()